In [46]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

import os
import warnings
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")


In [2]:
import sys
import pyspark.sql.functions as F
from src.config import data_paths, filter_map
from src.data_ingestion import DataLoader,DataMerger,Preprocessor
from src.logger import logger
from src.exception import M5Exception
from src.utils import spark

In [11]:
prod_cus_dim_path = data_paths['bronze']['processed_data']['product_customer_dim']
joined_data_path = data_paths['bronze']['processed_data']['joined_data']

In [26]:
data = pd.read_parquet(joined_data_path)
data = data.sort_values(by=['id','yearweek'])
data.head()

,id,yearweek,sales,sell_price,snap_CA_days,snap_TX_days,snap_WI_days,event_flag,f_event_stpatricksday,f_event_martinlutherkingday,...,f_event_independenceday,f_event_mothers_day,f_event_eidaladha,f_event_easter,f_event_superbowl,f_event_chanukah_end,f_event_memorialday,f_event_fathers_day,f_event_ramadan_starts,f_event_valentinesday
0,FOODS_1_001_CA_1,201104,3.0,2.0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,FOODS_1_001_CA_1,201105,7.0,2.0,5,3,3,1,0,0,...,0,0,0,0,1,0,0,0,0,0
2,FOODS_1_001_CA_1,201106,7.0,2.0,4,5,4,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,FOODS_1_001_CA_1,201107,6.0,2.0,0,1,2,1,0,0,...,0,0,0,0,0,0,0,0,0,1
4,FOODS_1_001_CA_1,201108,6.0,2.0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [27]:
prod_cus_dim = pd.read_parquet(prod_cus_dim_path)
prod_cus_dim.head()

,id,item_id,dept_id,cat_id,store_id,state_id
0,HOBBIES_1_035_CA_1,HOBBIES_1_035,HOBBIES_1,HOBBIES,CA_1,CA
1,HOUSEHOLD_1_076_CA_1,HOUSEHOLD_1_076,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA
2,HOUSEHOLD_1_222_CA_1,HOUSEHOLD_1_222,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA
3,HOUSEHOLD_1_284_CA_1,HOUSEHOLD_1_284,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA
4,HOUSEHOLD_1_309_CA_1,HOUSEHOLD_1_309,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA


In [28]:
joined_data = data.merge(prod_cus_dim,on=['id'],how='inner')
joined_data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6852789 entries, 0 to 6852788
Data columns (total 43 columns):
 #   Column                       Dtype   
---  ------                       -----   
 0   id                           object  
 1   yearweek                     object  
 2   sales                        float32 
 3   sell_price                   float32 
 4   snap_CA_days                 int8    
 5   snap_TX_days                 int8    
 6   snap_WI_days                 int8    
 7   event_flag                   int8    
 8   f_event_stpatricksday        int8    
 9   f_event_martinlutherkingday  int8    
 10  f_event_newyear              int8    
 11  f_event_christmas            int8    
 12  f_event_cinco_de_mayo        int8    
 13  f_event_pesach_end           int8    
 14  f_event_purim_end            int8    
 15  f_event_lentweek2            int8    
 16  f_event_eid_al_fitr          int8    
 17  f_event_laborday             int8    
 18  f_event_nbafinalsend  

In [29]:
#how many products[items], customers[stores], Categories?
joined_data['id'].nunique(), joined_data['item_id'].nunique(),joined_data['store_id'].nunique(),joined_data['cat_id'].nunique()

(30490, 3049, 10, 3)

In [66]:
joined_data.groupby(['state_id','cat_id'],as_index=False).agg(unique_ids=('id','nunique'),
                                          avg_sales = ('sales','mean'),
                                          total_sales = ('sales','sum'),
                                          sales_std = ('sales','std'),
                                          total_weeks = ('yearweek','nunique'))\
                                    .sort_values(by=['unique_ids','avg_sales'],ascending=False)\
                                    .assign(total_sales_m = lambda df:df['total_sales']/1e6,
                                            sales_pct = lambda df: (df['total_sales']/df['total_sales'].sum()) * 100,
                                            id_pct = lambda df: (df['unique_ids']/df['unique_ids'].sum())*100)

,state_id,cat_id,unique_ids,avg_sales,total_sales,sales_std,total_weeks,total_sales_m,sales_pct,id_pct
0,CA,FOODS,5748,14.267114,17940196.0,36.389175,279,17.940195,29.471031,18.852083
6,WI,FOODS,4311,12.734605,12123025.0,30.838293,279,12.123025,19.914948,14.139062
3,TX,FOODS,4311,12.247912,12116296.0,34.688515,279,12.116296,19.903894,14.139062
2,CA,HOUSEHOLD,4188,6.128171,5798715.0,13.465360,279,5.798715,9.525766,13.735651
5,TX,HOUSEHOLD,3141,5.456609,3918036.0,12.045346,279,3.918036,6.436305,10.301738
8,WI,HOUSEHOLD,3141,4.668539,3324597.0,10.914905,279,3.324597,5.461440,10.301738
1,CA,HOBBIES,2260,5.559175,2822588.0,12.878890,279,2.822588,4.636771,7.412266
4,TX,HOBBIES,1695,3.797665,1458759.0,7.611877,279,1.458759,2.396358,5.559200
7,WI,HOBBIES,1695,3.555218,1371788.0,7.582774,279,1.371788,2.253488,5.559200


- 67% of sales, 47% of series are from Foods
- 22% of sales, 34% of series are from HouseHold
- 11% of sales, 18% of series are from Hobbies

In [67]:

ID, CTRY, CAT, WEEK, Y, PRICE = "id", "state_id", "cat_id", "yearweek", "sales", "sell_price"
# SEAS_STRENGTH_MIN = 0.4   # STL seasonal strength threshold
# SEAS_CORR_MIN = 0.5       # year-over-year weekly profile correlation threshold


# def _nan_corr(X, Z, min_pairs):
#     """column-wise correlation ignoring NaN pairs"""
#     m = ~np.isnan(X) & ~np.isnan(Z)
#     X, Z = np.where(m, X, np.nan), np.where(m, Z, np.nan)
#     dx, dz = X - np.nanmean(X, 0), Z - np.nanmean(Z, 0)
#     den = np.sqrt(np.nansum(dx**2, 0) * np.nansum(dz**2, 0))
#     out = np.where(den > 0, np.nansum(dx * dz, 0) / den, np.nan)
#     return np.where(m.sum(0) >= min_pairs, out, np.nan)

# def _stl_seasonal_strength(x, period=52):
#     from statsmodels.tsa.seasonal import STL
#     x = pd.Series(x).interpolate().dropna()
#     if len(x) < 2 * period + 1:
#         return np.nan
#     try:
#         r = STL(np.log1p(x.to_numpy()), period=period, robust=True).fit()
#     except Exception:
#         return np.nan
#     return max(0.0, 1 - np.var(r.resid) / np.var(r.seasonal + r.resid))


# def _yearly_profile_corr(W):
#     """For each id: correlation of each year's weekly profile (weeks 1-52) with the
#     previous year's. Returns mean corr, number of year pairs, and share of pairs above threshold."""
#     L = np.log1p(W)
#     iso = L.index.isocalendar()
#     yr, wk = iso.year.to_numpy(), iso.week.to_numpy()
#     blocks = {}
#     for y in sorted(set(yr)):
#         m = (yr == y) & (wk <= 52)
#         blocks[y] = L.loc[m].set_axis(wk[m]).reindex(range(1, 53)).to_numpy()
#     ys = sorted(blocks)
#     cors = np.array([_nan_corr(blocks[b], blocks[a], min_pairs=40) for a, b in zip(ys[:-1], ys[1:])])
#     n_pairs = (~np.isnan(cors)).sum(0)
#     return (np.nanmean(cors, 0), n_pairs, np.where(n_pairs > 0, np.nansum(cors >= SEAS_CORR_MIN, 0) / np.maximum(n_pairs, 1), np.nan))



# def build_descriptors(df, stl=True):
#     df = df.copy()
#     df["date"] = pd.to_datetime(df[WEEK].astype(str) + "1", format="%G%V%u")  # ISO weeks
 
#     W = df.pivot(index="date", columns=ID, values=Y).sort_index()
#     W = W.reindex(pd.date_range(W.index.min(), W.index.max(), freq="7D"))
#     A = W.to_numpy(float)
 
#     obs = ~np.isnan(A)
#     pos = obs & (A > 0)
#     n, n_pos = obs.sum(0), pos.sum(0)
#     A_pos = np.where(pos, A, np.nan)
 
#     d = pd.DataFrame(index=W.columns)
#     d.index.name = ID
 
#     # size and history
#     d["history_len"] = n
#     d["mean_sales"] = np.nanmean(A, 0)
#     d["vol_52w"] = np.nansum(A[-52:], 0)
 
#     # intermittency
#     d["zero_share"] = 1 - n_pos / n
#     d["adi"] = np.where(n_pos > 0, n / np.maximum(n_pos, 1), np.inf)
#     d["cv2"] = (np.nanstd(A_pos, 0) / np.nanmean(A_pos, 0)) ** 2
 
#     # growth and lifecycle
#     l52, p52 = np.nanmean(A[-52:], 0), np.nanmean(A[-104:-52], 0)
#     d["yoy_ratio"] = np.where(p52 > 0, l52 / p52, np.nan)
#     d["tail_zero_weeks"] = np.cumprod(A[::-1] == 0, axis=0).sum(0)
 
#     # seasonality
#     L = np.log1p(A)
#     d["acf52"] = _nan_corr(L[52:], L[:-52], min_pairs=26)
#     d["yr_profile_corr"], d["n_year_pairs"], d["share_years_seasonal"] = _yearly_profile_corr(W)
#     if stl:
#         d["seasonal_strength"] = Parallel(n_jobs=-1, batch_size=256)(
#             delayed(_stl_seasonal_strength)(A[:, j]) for j in range(A.shape[1]))
#     else:
#         d["seasonal_strength"] = np.nan
 
#     # price
#     p = df[PRICE].where(df[PRICE] > 0)
#     g = df[ID]
#     d["price_cv"] = p.groupby(g).std() / p.groupby(g).mean()
#     d["promo_share"] = (p < 0.9 * p.groupby(g).transform("median")).groupby(g).mean()
 
#     # labels
#     d = d.join(df.groupby(ID)[[CTRY, CAT]].first())
#     d["is_new"] = d.history_len < 52
#     d["is_dead"] = d.tail_zero_weeks >= 13
#     d["demand_class"] = np.select(
#         [(d.adi < 1.32) & (d.cv2 < 0.49), (d.adi < 1.32) & (d.cv2 >= 0.49),
#          (d.adi >= 1.32) & (d.cv2 < 0.49)],
#         ["smooth", "erratic", "intermittent", ][:3], "lumpy")
 
#     # seasonal class: needs >= 2 years of history; seasonal only if BOTH the STL strength
#     # and the year-over-year profile correlation agree (and it is not an intermittent/dead series)
#     can_test = (d.history_len >= 104) & d.seasonal_strength.notna() & (d.n_year_pairs >= 1)
#     is_seas = (d.seasonal_strength >= SEAS_STRENGTH_MIN) & (d.yr_profile_corr >= SEAS_CORR_MIN)
#     d["seasonal_class"] = np.select([~can_test, is_seas], ["unknown", "seasonal"], "non_seasonal")
 
#     d["segment"] = d.demand_class + "_" + d.seasonal_class
 
#     # ABC within country x category on last-52-week volume
#     d = d.sort_values("vol_52w", ascending=False)
#     share = d.groupby([CTRY, CAT])["vol_52w"].transform(lambda s: s.cumsum() / s.sum())
#     d["abc"] = np.select([share <= 0.8, share <= 0.95], ["A", "B"], "C")
#     return d.sort_index()


SEAS_CORR_MIN = 0.5       # leave-one-year-out profile correlation (null sd ~0.2-0.3)
SEAS_AMP_MIN = 0.10       # seasonal swing in log units (~ +/-10%)


def _nan_corr(X, Z, min_pairs):
    """column-wise correlation ignoring NaN pairs"""
    m = ~np.isnan(X) & ~np.isnan(Z)
    X, Z = np.where(m, X, np.nan), np.where(m, Z, np.nan)
    dx, dz = X - np.nanmean(X, 0), Z - np.nanmean(Z, 0)
    den = np.sqrt(np.nansum(dx**2, 0) * np.nansum(dz**2, 0))
    out = np.where(den > 0, np.nansum(dx * dz, 0) / den, np.nan)
    return np.where(m.sum(0) >= min_pairs, out, np.nan)


def seasonal_features(W, min_years=3, min_buckets=9):
    """Cheap, trend-robust seasonality test (no model fitting).

    1. log1p(sales) minus centred 52-week moving average  -> removes level and trend
    2. average into 13 four-week buckets per ISO year      -> removes weekly noise
    3. leave-one-year-out: correlate each year's bucket profile with the average of the
       other years, then average over years                -> "does last year's shape repeat?"
    4. amplitude = std of the average profile (log units)  -> "is the swing big enough to matter?"
    """
    L = np.log1p(W)
    ma = L.rolling(52).mean()
    x = L - (ma.shift(-25) + ma.shift(-26)) / 2          # centred 2x52 moving average

    iso = x.index.isocalendar()
    yr = iso.year.to_numpy(dtype=int)
    bucket = np.minimum((iso.week.to_numpy(dtype=int) - 1) // 4, 12)
    P = x.groupby([yr, bucket]).mean().where(x.notna().groupby([yr, bucket]).sum() >= 3)
    ys = sorted(set(yr))
    arr = np.stack([P.loc[y].reindex(range(13)).to_numpy() for y in ys])   # years x 13 x N
    arr = arr - np.nanmean(arr, axis=1, keepdims=True)
    valid = (~np.isnan(arr)).sum(1) >= min_buckets                          # years x N
    arr = np.where(valid[:, None, :], arr, np.nan)

    cors = []
    for i in range(len(ys)):
        others = np.nanmean(np.delete(arr, i, 0), axis=0)
        c = _nan_corr(arr[i], others, min_pairs=min_buckets)
        cors.append(np.where(np.delete(valid, i, 0).sum(0) >= 2, c, np.nan))
    out = pd.DataFrame(index=W.columns)
    out["n_seas_years"] = valid.sum(0)
    out["seas_corr"] = np.nanmean(np.array(cors), axis=0)
    out["seas_amp"] = np.nanstd(np.nanmean(arr, axis=0), axis=0)
    return out


def build_descriptors(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df[WEEK].astype(str) + "1", format="%G%V%u")  # ISO weeks

    W = df.pivot(index="date", columns=ID, values=Y).sort_index()
    W = W.reindex(pd.date_range(W.index.min(), W.index.max(), freq="7D"))
    A = W.to_numpy(float)

    obs = ~np.isnan(A)
    pos = obs & (A > 0)
    n, n_pos = obs.sum(0), pos.sum(0)
    A_pos = np.where(pos, A, np.nan)

    d = pd.DataFrame(index=W.columns)
    d.index.name = ID

    # size and history
    d["age_weeks"] = n
    d["mean_sales"] = np.nanmean(A, 0)
    d["vol_52w"] = np.nansum(A[-52:], 0)

    # intermittency
    d["zero_share"] = 1 - n_pos / n
    d["adi"] = np.where(n_pos > 0, n / np.maximum(n_pos, 1), np.inf)
    d["cv2"] = (np.nanstd(A_pos, 0) / np.nanmean(A_pos, 0)) ** 2

    # growth and lifecycle
    l52, p52 = np.nanmean(A[-52:], 0), np.nanmean(A[-104:-52], 0)
    d["yoy_ratio"] = np.where(p52 > 0, l52 / p52, np.nan)
    d["tail_zero_weeks"] = np.cumprod(A[::-1] == 0, axis=0).sum(0)

    # seasonality
    L = np.log1p(A)
    d["acf52"] = _nan_corr(L[52:], L[:-52], min_pairs=26)
    d = d.join(seasonal_features(W))

    # price
    p = df[PRICE].where(df[PRICE] > 0)
    g = df[ID]
    d["price_cv"] = p.groupby(g).std() / p.groupby(g).mean()
    d["promo_share"] = (p < 0.9 * p.groupby(g).transform("median")).groupby(g).mean()

    # labels
    d = d.join(df.groupby(ID)[[CTRY, CAT]].first())
    d["is_new"] = d.age_weeks < 52
    d["is_dead"] = d.tail_zero_weeks >= 13
    d["demand_class"] = np.select(
        [(d.adi < 1.32) & (d.cv2 < 0.49), (d.adi < 1.32) & (d.cv2 >= 0.49),
         (d.adi >= 1.32) & (d.cv2 < 0.49)],
        ["smooth", "erratic", "intermittent"], "lumpy")

    # seasonal class: needs >= 3 usable years; seasonal = shape repeats AND swing is material
    d["seasonal_class"] = np.select(
        [d.n_seas_years < 3, (d.seas_corr >= SEAS_CORR_MIN) & (d.seas_amp >= SEAS_AMP_MIN)],
        ["unknown", "seasonal"], "non_seasonal")

    d["segment"] = d.demand_class + "_" + d.seasonal_class

    # ABC within country x category on last-52-week volume
    d = d.sort_values("vol_52w", ascending=False)
    share = d.groupby([CTRY, CAT])["vol_52w"].transform(lambda s: s.cumsum() / s.sum())
    d["abc"] = np.select([share <= 0.8, share <= 0.95], ["A", "B"], "C")
    return d.sort_index()

 

In [68]:
data_cube = build_descriptors(joined_data)

In [69]:
data_cube

,age_weeks,mean_sales,vol_52w,zero_share,adi,cv2,yoy_ratio,tail_zero_weeks,acf52,n_seas_years,...,price_cv,promo_share,state_id,cat_id,is_new,is_dead,demand_class,seasonal_class,segment,abc
id,,,,,,,,,,,,,,,,,,,,,
FOODS_1_001_CA_1,279,4.817204,97.0,0.100358,1.111554,0.439299,1.224272,1,-0.096013,4,...,0.050684,0.297491,CA,FOODS,False,False,smooth,non_seasonal,smooth_non_seasonal,B
FOODS_1_001_CA_2,279,7.308244,116.0,0.103943,1.116000,0.431322,1.133835,1,-0.026740,4,...,0.050879,0.301075,CA,FOODS,False,False,smooth,non_seasonal,smooth_non_seasonal,B
FOODS_1_001_CA_3,279,7.677419,135.0,0.100358,1.111554,0.574219,1.070122,0,-0.005103,4,...,0.054715,0.336918,CA,FOODS,False,False,erratic,non_seasonal,erratic_non_seasonal,B
FOODS_1_001_CA_4,279,2.186380,27.0,0.222222,1.285714,0.425284,0.723711,0,0.105158,4,...,0.050684,0.297491,CA,FOODS,False,False,smooth,non_seasonal,smooth_non_seasonal,C
FOODS_1_001_TX_1,279,3.691756,80.0,0.168459,1.202586,0.672044,1.168539,0,-0.192193,4,...,0.075806,0.308244,TX,FOODS,False,False,erratic,non_seasonal,erratic_non_seasonal,B
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
HOUSEHOLD_2_516_TX_2,279,1.494624,28.0,0.236559,1.309859,0.387834,0.735354,0,0.074019,4,...,0.035885,0.007168,TX,HOUSEHOLD,False,False,smooth,non_seasonal,smooth_non_seasonal,C
HOUSEHOLD_2_516_TX_3,279,0.720430,19.0,0.498208,1.992857,0.202445,0.950000,1,0.058055,4,...,0.070852,0.007168,TX,HOUSEHOLD,False,False,intermittent,non_seasonal,intermittent_non_seasonal,C
HOUSEHOLD_2_516_WI_1,278,0.528777,12.0,0.575540,2.355932,0.184969,1.950000,1,0.053553,4,...,0.000000,0.000000,WI,HOUSEHOLD,False,False,intermittent,non_seasonal,intermittent_non_seasonal,C


In [70]:
data_cube.groupby(['state_id','cat_id']).agg(
                                            delisted_id_count = ('is_dead','sum'),
                                            npd_id_count = ('is_new','sum')
                                             )

delisted_id_count  npd_id_count
state_id cat_id                                    
CA       FOODS                      0            95
         HOBBIES                    0             6
         HOUSEHOLD                  0            13
TX       FOODS                      0            20
         HOBBIES                    0             1
         HOUSEHOLD                  0            11
WI       FOODS                      0            17
         HOBBIES                    0             3
         HOUSEHOLD                  0             9

In [71]:
data_cube.groupby(["state_id", "cat_id", "demand_class"]).size().unstack()

demand_class        erratic  intermittent  lumpy  smooth
state_id cat_id                                         
CA       FOODS          739          1304    638    3067
         HOBBIES        152           878    297     933
         HOUSEHOLD      215          1546    291    2136
TX       FOODS          713          1002    841    1755
         HOBBIES        159           695    292     549
         HOUSEHOLD      202          1282    271    1386
WI       FOODS          680          1151    649    1831
         HOBBIES        131           794    251     519
         HOUSEHOLD      179          1426    269    1267

In [72]:
data_cube.groupby(["state_id", "cat_id", "seasonal_class"]).size().unstack()

seasonal_class      non_seasonal  seasonal  unknown
state_id cat_id                                    
CA       FOODS              3498       345     1905
         HOBBIES            1495       115      650
         HOUSEHOLD          2671       274     1243
TX       FOODS              2889       240     1182
         HOBBIES            1147        96      452
         HOUSEHOLD          2071       213      857
WI       FOODS              2747       285     1279
         HOBBIES            1152        92      451
         HOUSEHOLD          2033       235      873

In [73]:
data_cube.groupby(["state_id", "cat_id", "segment"]).size().unstack()

segment             erratic_non_seasonal  erratic_seasonal  erratic_unknown  \
state_id cat_id                                                               
CA       FOODS                       479                52              208   
         HOBBIES                     106                13               33   
         HOUSEHOLD                   129                25               61   
TX       FOODS                       508                56              149   
         HOBBIES                     103                17               39   
         HOUSEHOLD                   133                18               51   
WI       FOODS                       491                45              144   
         HOBBIES                      94                 6               31   
         HOUSEHOLD                   125                15               39   

segment             intermittent_non_seasonal  intermittent_seasonal  \
state_id cat_id                                                        
CA       FOODS                            648                     91   
         HOBBIES                          548                     36   
         HOUSEHOLD                        881                    136   
TX       FOODS                            580                     48   
         HOBBIES                          459                     25   
         HOUSEHOLD                        813                     94   
WI       FOODS                            638                     86   
         HOBBIES                          503                     42   
         HOUSEHOLD                        920                    100   

segment             intermittent_unknown  lumpy_non_seasonal  lumpy_seasonal  \
state_id cat_id                                                                
CA       FOODS                       565                 388              50   
         HOBBIES                     294                 169              30   
         HOUSEHOLD                   529                 171              35   
TX       FOODS                       374                 545              70   
         HOBBIES                     211                 161              34   
         HOUSEHOLD                   375                 164              31   
WI       FOODS                       427                 406              57   
         HOBBIES                     249                 142              31   
         HOUSEHOLD                   406                 145              52   

segment             lumpy_unknown  smooth_non_seasonal  smooth_seasonal  \
state_id cat_id                                                           
CA       FOODS                200                 1983              152   
         HOBBIES               98                  672               36   
         HOUSEHOLD             85                 1490               78   
TX       FOODS                226                 1256               66   
         HOBBIES               97                  424               20   
         HOUSEHOLD             76                  961               70   
WI       FOODS                186                 1212               97   
         HOBBIES               78                  413               13   
         HOUSEHOLD             72                  843               68   

segment             smooth_unknown  
state_id cat_id                     
CA       FOODS                 932  
         HOBBIES               225  
         HOUSEHOLD             568  
TX       FOODS                 433  
         HOBBIES               105  
         HOUSEHOLD             355  
WI       FOODS                 522  
         HOBBIES                93  
         HOUSEHOLD             356

In [63]:
data_cube.groupby("demand_class",as_index=False)["vol_52w"].sum().sort_values(by=['vol_52w'],ascending=False)

,demand_class,vol_52w
3,smooth,3335769.0
1,intermittent,855565.0
0,erratic,637660.0
2,lumpy,327310.0
